In [ ]:
# ==============================
# IMPORTS
# ==============================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"  # use "browser" se não estiver em notebook


# ==============================
# FUNÇÕES AUXILIARES
# ==============================
def aplicar_layout_padrao(fig):
    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        legend_title_text="",
        margin=dict(t=60, l=40, r=40, b=40)
    )


def reforco_visual_barras(fig, casas=2, texto_pos="outside"):
    fig.update_traces(
        texttemplate="%{y}",
        textposition=texto_pos,
        cliponaxis=False
    )
    fig.update_yaxes(tickformat=f".{casas}f")


# ==============================
# LEITURA DOS DADOS
# ==============================
try:
    logs = pd.read_csv("logs_atendimento_real_tese.csv")
    kpi_cenario = pd.read_csv("kpi_por_cenario.csv")
    kpi_prioridade = pd.read_csv("kpi_por_prioridade.csv")
    kpi_pontuacao = pd.read_csv("kpi_por_pontuacao_prioridade.csv")
except FileNotFoundError as e:
    raise FileNotFoundError(f"Ficheiro não encontrado: {e.filename}")

colunas_logs_necessarias = {
    "estado_final", "instante_entrada", "instante_chamada",
    "instante_fim", "cenario", "tipo_atendimento", "id_item_fila"
}

faltantes = colunas_logs_necessarias - set(logs.columns)
if faltantes:
    raise ValueError(f"Colunas em falta no logs: {faltantes}")


# ==============================
# FILTRO DE ATENDIMENTOS CONCLUÍDOS
# ==============================
logs_done = logs[logs["estado_final"] == "DONE"].copy()

for col in ["instante_entrada", "instante_chamada", "instante_fim"]:
    if not pd.api.types.is_datetime64_any_dtype(logs_done[col]):
        logs_done[col] = pd.to_datetime(logs_done[col], errors="coerce")

logs_done = logs_done.dropna(subset=["instante_entrada", "instante_chamada", "instante_fim"])

logs_done = logs_done[
    (logs_done["instante_chamada"] >= logs_done["instante_entrada"]) &
    (logs_done["instante_fim"] >= logs_done["instante_chamada"])
]


# ==============================
# CÁLCULO DE MÉTRICAS
# ==============================
logs_done["espera_min_calc"] = (
    logs_done["instante_chamada"] - logs_done["instante_entrada"]
).dt.total_seconds() / 60

logs_done["atendimento_min_calc"] = (
    logs_done["instante_fim"] - logs_done["instante_chamada"]
).dt.total_seconds() / 60


metricas = (
    logs_done
    .groupby("cenario")
    .agg(
        total_atendidos=("id_item_fila", "count"),
        espera_media_min=("espera_min_calc", "mean"),
        espera_p95_min=("espera_min_calc", lambda x: x.quantile(0.95, interpolation="linear")),
        atendimento_medio_min=("atendimento_min_calc", "mean")
    )
    .reset_index()
)


# ==============================
# GRÁFICO 1 — TOTAL ATENDIDOS
# ==============================
fig1 = px.bar(
    metricas,
    x="cenario",
    y="total_atendidos",
    title="Total de Itens Atendidos por Cenário"
)

aplicar_layout_padrao(fig1)
reforco_visual_barras(fig1, casas=0)
fig1.show()


# ==============================
# GRÁFICO 2 — ESPERA MÉDIA
# ==============================
fig2 = px.bar(
    metricas,
    x="cenario",
    y="espera_media_min",
    title="Tempo Médio de Espera (min) por Cenário"
)

aplicar_layout_padrao(fig2)
reforco_visual_barras(fig2)
fig2.show()


# ==============================
# GRÁFICO 3 — ESPERA P95
# ==============================
fig3 = px.bar(
    metricas,
    x="cenario",
    y="espera_p95_min",
    title="Tempo de Espera P95 (min) por Cenário"
)

aplicar_layout_padrao(fig3)
reforco_visual_barras(fig3)
fig3.show()


# ==============================
# GRÁFICO 4 — TEMPO MÉDIO DE ATENDIMENTO
# ==============================
fig4 = px.bar(
    metricas,
    x="cenario",
    y="atendimento_medio_min",
    title="Tempo Médio de Atendimento (min) por Cenário"
)

aplicar_layout_padrao(fig4)
reforco_visual_barras(fig4)
fig4.show()
